## Process - CNAF export of 2026-10-06, AEEH
- Load CNAF file
- Clean data and first mapping for bdd injection
- Drop duplicates
- Add default column values
- Select the AEEH eligibility route
- Output to CSV

In [ ]:
import os
import csv
import sys
from pathlib import Path

from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib itself sits one level up, in partners/, next to the other partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

# All the DataFrame processing lives in two modules so it can be unit tested: what every
# partner routed through qf-batch shares in ../partners_lib.py, and what is specific to
# the CNAF file (its raw columns, its address lines) in clean_cnaf_lib.py.
import partners_lib as partners
import clean_cnaf_lib as cnaf

load_dotenv()

cnaf_input_filepath = os.environ['CNAF_PATHFILE_10_06_2026']
base_output_filepath = os.environ['DB_CNAF_EXPORT_2026_AEEH']

In [ ]:
# CNAF - no PASSPORT metadata row in this export, only the header row (skipped: it repeats
# 'presta', so columns are still supplied positionally)
cnaf_df = cnaf.read_raw_cnaf_csv(
    cnaf_input_filepath, columns=cnaf.CNAF_RAW_COLUMNS_10_06_2026, skiprows=1)

In [ ]:
# clean white spaces within all columns - no trailing invalid row in this export
cnaf_df = partners.strip_all_string_columns(cnaf_df)

In [ ]:
# Explode postal code & commune from initial column containing both
cnaf_df = cnaf.split_postal_code_and_commune(cnaf_df)

In [ ]:
# Clean extra white spaces
cnaf_df = cnaf.normalize_full_name_spacing(cnaf_df)

In [ ]:
# map CNAF columns to the PSP schema (see cnaf.CNAF_COLUMN_MAPPING_10_06_2026)
df_psp_mapped_cnaf = cnaf.map_cnaf_columns(cnaf_df, cnaf.CNAF_COLUMN_MAPPING_10_06_2026)

print(df_psp_mapped_cnaf['situation_origine'].value_counts(dropna=False))

# this file must contain only AEEH
assert (df_psp_mapped_cnaf['droit_aeeh_juillet'] != '').sum() == len(df_psp_mapped_cnaf)

In [ ]:
# Allocataire missing phone number
df_psp_mapped_cnaf = partners.clear_placeholder_phone_numbers(df_psp_mapped_cnaf)

# Allocataire's qualite
df_psp_mapped_cnaf = partners.normalize_allocataire_qualite(df_psp_mapped_cnaf)

# Additionnal address details & allocataire's street address
df_psp_mapped_cnaf = cnaf.build_allocataire_address_fields(df_psp_mapped_cnaf)

# Organism & situation - every row is AEEH (asserted above)
df_psp_mapped_cnaf['organisme'] = 'CAF'
df_psp_mapped_cnaf['situation'] = 'AEEH'

In [ ]:
# Format date_naissance to datetime python object for processing
df_psp_mapped_cnaf = partners.parse_beneficiary_birthdate(df_psp_mapped_cnaf)

In [ ]:
# remove unused, including the CNAF rights and quotient familial columns
df_psp_mapped_cnaf = cnaf.drop_raw_columns(
    df_psp_mapped_cnaf, cnaf.RAW_COLUMNS_TO_DROP_10_06_2026)

In [ ]:
# remove rows with missing necessary values (if one of those value are missing we cannot
# generate a code), then columns with all null value
df_valid = partners.filter_rows_missing_required_fields(df_psp_mapped_cnaf)

In [ ]:
# Upper case these columns for the merge
df_valid = partners.normalize_identity_casing(df_valid)

In [ ]:
# lower case on emails on all
df_valid = partners.normalize_email_casing(df_valid)

In [ ]:
# Preliminary filter on AEEH's lower bound, ahead of the full AEEH window below.
df_valid_after = partners.filter_within_eligibility_floor(df_valid, floor_date=partners.AEEH_DOB_MIN)

print(f"{len(df_valid) - len(df_valid_after)} rows removed because they are born before the AEEH window")

In [ ]:
# add missing 0 to phone numbers, and set '0' phone values to None
df_valid_after = partners.fix_phone_number_formatting(df_valid_after)

In [ ]:
# set Nan values for not existing courriel
df_valid_after = partners.clear_blank_email(df_valid_after)

In [ ]:
# add 4h on all birthdates
df_valid_after = partners.shift_birthdate_by_hours(df_valid_after)

In [ ]:
# remove duplicate beneficiaries (see partners.DEDUPLICATION_KEY_COLUMNS)
df_valid_no_duplicate, duplicate_count = partners.drop_duplicate_beneficiaries(df_valid_after)

print(f"{duplicate_count} duplicate rows were removed")

In [ ]:
# map allocataire json
df_valid_no_duplicate = partners.add_allocataire_json_column(df_valid_no_duplicate)

In [ ]:
# map adresse_allocataire json
df_valid_no_duplicate = partners.add_adresse_allocataire_json_column(df_valid_no_duplicate)

In [ ]:
## drop the columns now folded into the JSON columns, plus the qf-batch pivot-only ones
## (see partners.FINAL_COLUMNS_TO_DROP). `situation` and `date_naissance`, which the AEEH
## route below needs, are not part of that list and stay on df_final.
df_final = partners.drop_intermediate_columns(df_valid_no_duplicate)

In [ ]:
# AEEH route: 6-19 ans révolus - no quotient_familial call needed, CNAF already flags it
aeeh_index = partners.aeeh_eligible_index(df_final)
df_final_aeeh = partners.select_eligible_by_index(df_final, aeeh_index).reset_index(drop=True)

print(f"{len(df_final) - len(df_final_aeeh)} rows removed because they are born after the AEEH window")

In [ ]:
# Cast to string
df_final_aeeh.loc[:,'date_naissance'] = df_final_aeeh['date_naissance'].astype(str)

In [ ]:
# output to CSV files
df_final_aeeh.to_csv(base_output_filepath, sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

In [ ]:
print(f"{len(df_final_aeeh)} aeeh written to {base_output_filepath}")

In [ ]:
# Recap of the rows removed at each step
with open(cnaf_input_filepath, encoding='utf-8-sig') as raw_file:
    raw_row_count = sum(1 for _ in raw_file) - 1  # minus the header row

removed_rows = {
    'lignes mal formées ignorées à la lecture (cellule 2)': raw_row_count - len(df_psp_mapped_cnaf),
    'nom/prenom/date_naissance/genre manquant (cellule 10)': len(df_psp_mapped_cnaf) - len(df_valid),
    f"nés avant le {partners.AEEH_DOB_MIN:%d/%m/%Y}, borne basse AEEH (cellule 13)": len(df_valid) - len(df_valid_after),
    'doublons (cellule 17)': duplicate_count,
    f"nés après le {partners.AEEH_DOB_MAX:%d/%m/%Y}, borne haute AEEH (cellule 21)": len(df_final) - len(df_final_aeeh),
}

print(f"{raw_row_count} lignes dans le fichier en entrée")
for step, count in removed_rows.items():
    print(f"  - {count} {step}")
print(f"{sum(removed_rows.values())} lignes supprimées au total, {len(df_final_aeeh)} écrites")
assert raw_row_count - sum(removed_rows.values()) == len(df_final_aeeh)

## ⏭ Next: 10_06_2026_cnaf_aeeh_deduplication.ipynb
It drops the beneficiaries LAMP already holds, then `../generate_new_codes.ipynb` with
`SOURCE = 'CNAF_AEEH'` assigns the remaining ones their `id_psp`.